In [1]:
!pip install cvxpy control --quiet
print('OK')

OK



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\PC\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
!pip install casadi


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\PC\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
"""
MHE + NMPC en Lazo Cerrado — Simulación Parámetros Planta Real
===============================================================
Migrado desde MATLAB a Python para uniformidad con el resto de códigos.

Parámetros físicos calibrados experimentalmente:
  - Atank = 3.02e-3 m²  (tanque PVC cilíndrico D=6.2cm)
  - αo = 0.0271, Do = 10mm  (calibrado por vaciado libre)
  - qi1max=2.69e-5, qi3max=2.41e-5 m³/s  (bombas R385 al 70%)
  - Referencias: 0.10 → 0.20 → 0.15 m
  - σ = 1 mm  (ruido HC-SR04)
  - rng(42) para reproducibilidad
  - Estado máximo: 0.27 m (h1_ss ≈ 0.221 m a h2=0.20 m)

Estructura:
  - MHE  (N=20): estima h1, h2, h3 desde mediciones ruidosas de h1, h3
  - NMPC (N=20): controla h2 usando el estado estimado por el MHE
  - Ambos usan IPOPT via CasADi, igual que la versión MATLAB
  - Métricas RMSE_SS con exclusión de 60s post-cambio de referencia
  - Al final imprime tabla comparativa con Koopman Closed-Loop
"""

import numpy as np
from scipy.optimize import least_squares
import casadi as ca
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
import time

warnings.filterwarnings('ignore')
output_dir = Path('./outputs_sim')
output_dir.mkdir(parents=True, exist_ok=True)

# ============================================================================
# PARÁMETROS FÍSICOS — PLANTA REAL CALIBRADA
# ============================================================================
P_NOM = {
    'Atank'   : 3.02e-3,
    'rho'     : 997.0,    'eta'      : 8.9e-4,   'g'        : 9.81,
    'alphao1' : 0.0271,   'Do1'      : 10e-3,
    'alphao2' : 0.0271,   'A2'       : 7.85e-5,
    'alphao3' : 0.0271,   'Do3'      : 10e-3,
    'alpha120': 0.3038,   'D12'      : 10e-3,
    'A12'     : 7.85e-5,  'lambdac12': 24000,
    'alpha230': 0.1344,   'D23'      : 10e-3,
    'A23'     : 7.85e-5,  'lambdac23': 29600,
    'qi1max'  : 2.69e-5,
    'qi3max'  : 2.41e-5,
}

Ts         = 2.0
T_sim      = 800
Nsim       = int(T_sim / Ts)
sigma      = 1e-3          # ruido HC-SR04 (1 mm)
SETTLING_S = 60
EPS        = 1e-10
EPS_S      = 1e-8
H_MAX      = 0.27         # margen físico real (saturation integrator RK4)

# ============================================================================
# MODELO PYTHON
# ============================================================================
def nl_ode(x, u, p=P_NOM):
    h1 = max(x[0], EPS); h2 = max(x[1], EPS); h3 = max(x[2], EPS)
    qi1 = p['qi1max'] * np.clip(u[0], 0, 1)
    qi3 = p['qi3max'] * np.clip(u[1], 0, 1)
    qo1 = p['alphao1']*(p['Do1']**2*np.pi/4)*np.sqrt(2*p['g']*h1)
    qo2 = p['alphao2']*p['A2']              *np.sqrt(2*p['g']*h2)
    qo3 = p['alphao3']*(p['Do3']**2*np.pi/4)*np.sqrt(2*p['g']*h3)
    dh12 = h1-h2
    l12  = p['D12']*p['rho']/p['eta']*np.sqrt(2*p['g']*abs(dh12)+EPS)
    a12  = p['alpha120']*np.tanh(2*l12/p['lambdac12'])
    q12  = a12*p['A12']*np.sqrt(2*p['g']*abs(dh12)+EPS)*np.sign(dh12+1e-15)
    dh23 = h2-h3
    l23  = p['D23']*p['rho']/p['eta']*np.sqrt(2*p['g']*abs(dh23)+EPS)
    a23  = p['alpha230']*np.tanh(2*l23/p['lambdac23'])
    q23  = a23*p['A23']*np.sqrt(2*p['g']*abs(dh23)+EPS)*np.sign(dh23+1e-15)
    return np.array([
        (qi1-q12-qo1)/p['Atank'],
        (q12-q23-qo2)/p['Atank'],
        (qi3+q23-qo3)/p['Atank'],
    ])

def rk4_py(x, u, p=P_NOM, dt=Ts):
    k1 = nl_ode(x,u,p); k2 = nl_ode(x+dt/2*k1,u,p)
    k3 = nl_ode(x+dt/2*k2,u,p); k4 = nl_ode(x+dt*k3,u,p)
    return np.clip(x + dt/6*(k1+2*k2+2*k3+k4), 0, H_MAX)

def compute_ss(h2_ref, p=P_NOM):
    def eqs(v):
        h1, h3, u1 = v
        return nl_ode([max(h1,1e-4), h2_ref, max(h3,1e-4)],
                      [np.clip(u1,0,1), 0.0], p)
    # Guesses: primero cercanos al SS real (h1_ss ≈ 1.1*h2_ref), luego más alejados
    guesses = [
        (h2_ref*1.1, h2_ref*0.9, 0.5),
        (h2_ref*1.1, h2_ref*0.8, 0.6),
        (h2_ref*1.2, h2_ref*0.8, 0.55),
        (h2_ref*1.4, h2_ref*0.8, 0.5),
        (h2_ref*1.6, h2_ref*0.7, 0.7),
        (h2_ref*1.8, h2_ref*0.8, 0.85),
        (h2_ref*1.8, h2_ref*0.85, 0.85),
    ]
    best, best_res = None, 1e10
    for g in guesses:
        try:
            s = least_squares(eqs, list(g),
                              bounds=([0.01, 0.01, 0.0], [H_MAX, H_MAX, 1.0]),
                              max_nfev=5000)
            if s.success and np.linalg.norm(s.fun) < best_res:
                best_res = np.linalg.norm(s.fun); best = s.x
        except: pass
    if best is not None and best_res < 1e-4:
        return np.array([best[0], h2_ref, best[1]]), np.array([np.clip(best[2],0,1), 0.0])
    return None, None

# ============================================================================
# MODELO CASADI COMPARTIDO
# ============================================================================
def build_casadi_model(p=P_NOM):
    nx, nu = 3, 2
    x_s = ca.SX.sym('x', nx); u_s = ca.SX.sym('u', nu)
    h1, h2, h3 = x_s[0], x_s[1], x_s[2]

    qi1 = p['qi1max'] * u_s[0]
    qi3 = p['qi3max'] * u_s[1]
    h1p = (h1+ca.sqrt(h1**2+EPS_S))/2
    h2p = (h2+ca.sqrt(h2**2+EPS_S))/2
    h3p = (h3+ca.sqrt(h3**2+EPS_S))/2

    qo1 = p['alphao1']*(p['Do1']**2*np.pi/4)*ca.sqrt(2*p['g']*h1p)
    qo2 = p['alphao2']*p['A2']              *ca.sqrt(2*p['g']*h2p)
    qo3 = p['alphao3']*(p['Do3']**2*np.pi/4)*ca.sqrt(2*p['g']*h3p)

    dh12=h1-h2; abs12=ca.sqrt(dh12**2+EPS_S)
    l12=p['D12']*p['rho']/p['eta']*ca.sqrt(2*p['g']*abs12)
    a12=p['alpha120']*ca.tanh(2*l12/p['lambdac12'])
    q12=a12*p['A12']*ca.sqrt(2*p['g']*abs12)*ca.sign(dh12)

    dh23=h2-h3; abs23=ca.sqrt(dh23**2+EPS_S)
    l23=p['D23']*p['rho']/p['eta']*ca.sqrt(2*p['g']*abs23)
    a23=p['alpha230']*ca.tanh(2*l23/p['lambdac23'])
    q23=a23*p['A23']*ca.sqrt(2*p['g']*abs23)*ca.sign(dh23)

    xdot = ca.vertcat(
        (qi1-q12-qo1)/p['Atank'],
        (q12-q23-qo2)/p['Atank'],
        (qi3+q23-qo3)/p['Atank'],
    )
    f_cas = ca.Function('f', [x_s, u_s], [xdot])
    k1r=f_cas(x_s,u_s); k2r=f_cas(x_s+Ts/2*k1r,u_s)
    k3r=f_cas(x_s+Ts/2*k2r,u_s); k4r=f_cas(x_s+Ts*k3r,u_s)
    F_rk4 = ca.Function('F', [x_s,u_s], [x_s+Ts/6*(k1r+2*k2r+2*k3r+k4r)])
    return F_rk4, nx, nu

# ============================================================================
# CONSTRUCCIÓN MHE
# ============================================================================
def build_mhe(F_rk4, nx, nu):
    print("  Construyendo MHE...")
    N_mhe = 20; ny = 2
    S_mhe = np.diag([50.0, 0.5, 50.0])
    Q_mhe = np.diag([10.0, 50.0, 10.0])
    R_mhe = np.diag([1.0, 1.0])
    C_mat = np.array([[1,0,0],[0,0,1]], dtype=float)
    xmin  = np.array([0.0, 0.0, 0.0]); xmax = np.array([H_MAX, H_MAX, H_MAX])

    X_mhe  = ca.SX.sym('X', nx, N_mhe+1)
    xbar_p = ca.SX.sym('xbar', nx)
    Uk_p   = ca.SX.sym('Uk', nu, N_mhe)
    Yk_p   = ca.SX.sym('Yk', ny, N_mhe)

    J = 0
    e0 = X_mhe[:,0] - xbar_p
    J += ca.mtimes(e0.T, ca.DM(S_mhe) @ e0)

    for j in range(N_mhe):
        xj  = X_mhe[:,j]; xj1 = X_mhe[:,j+1]
        uj  = Uk_p[:,j];  yj  = Yk_p[:,j]
        wj  = xj1 - F_rk4(xj, uj)
        vj  = yj - ca.DM(C_mat) @ xj
        J  += ca.mtimes(wj.T, ca.DM(Q_mhe) @ wj)
        J  += ca.mtimes(vj.T, ca.DM(R_mhe) @ vj)

    w_mhe   = ca.reshape(X_mhe, nx*(N_mhe+1), 1)
    par_mhe = ca.vertcat(xbar_p,
                         ca.reshape(Uk_p, nu*N_mhe, 1),
                         ca.reshape(Yk_p, ny*N_mhe, 1))
    lbw = np.tile(xmin, N_mhe+1)
    ubw = np.tile(xmax, N_mhe+1)

    nlp  = {'x': w_mhe, 'f': J, 'p': par_mhe}
    opts = {'ipopt.print_level': 0, 'ipopt.max_iter': 200,
            'ipopt.tol': 1e-4, 'print_time': 0}
    solver_mhe = ca.nlpsol('solver_mhe', 'ipopt', nlp, opts)
    print(f"  MHE listo (N={N_mhe}).")
    return solver_mhe, N_mhe, ny, C_mat, xmin, xmax, lbw, ubw

# ============================================================================
# CONSTRUCCIÓN NMPC
# ============================================================================
def build_nmpc(F_rk4, nx, nu):
    print("  Construyendo NMPC...")
    N   = 20; q = 500
    R1  = np.diag([0.1, 1.0]); R2 = np.diag([0.1, 0.1])
    cT  = np.array([0.0, 1.0, 0.0])
    umin = np.array([0.0,0.0]); umax = np.array([1.0,1.0])
    xmin = np.array([0.0, 0.0, 0.0]); xmax = np.array([H_MAX, H_MAX, H_MAX])

    X_d  = ca.SX.sym('X', nx, N+1); U_d = ca.SX.sym('U', nu, N)
    x0_p = ca.SX.sym('x0', nx); yr_p = ca.SX.sym('yr'); up_p = ca.SX.sym('up', nu)

    J = 0; g = []; lbg = []; ubg = []
    g += [X_d[:,0]-x0_p]; lbg += [0.0]*nx; ubg += [0.0]*nx

    for k in range(N):
        g   += [X_d[:,k+1] - F_rk4(X_d[:,k], U_d[:,k])]
        lbg += [0.0]*nx; ubg += [0.0]*nx
        yk  = ca.dot(ca.DM(cT), X_d[:,k+1])
        ey  = yk - yr_p
        du  = U_d[:,0]-up_p if k==0 else U_d[:,k]-U_d[:,k-1]
        wk  = 10 if k == N-1 else 1
        J  += wk*q*ey**2
        J  += ca.mtimes(U_d[:,k].T, ca.DM(R1)@U_d[:,k])
        J  += ca.mtimes(du.T, ca.DM(R2)@du)

    w_mpc   = ca.vertcat(ca.reshape(X_d,nx*(N+1),1), ca.reshape(U_d,nu*N,1))
    par_mpc = ca.vertcat(x0_p, yr_p, up_p)
    lbw = np.concatenate([np.tile(xmin,N+1), np.tile(umin,N)])
    ubw = np.concatenate([np.tile(xmax,N+1), np.tile(umax,N)])

    nlp  = {'x': w_mpc, 'f': J, 'g': ca.vertcat(*g), 'p': par_mpc}
    opts = {'ipopt.print_level': 0, 'ipopt.max_iter': 500,
            'ipopt.tol': 1e-5, 'ipopt.acceptable_tol': 1e-4, 'print_time': 0}
    solver_mpc = ca.nlpsol('solver_mpc', 'ipopt', nlp, opts)
    print(f"  NMPC listo (N={N}, q={q}).")
    return solver_mpc, N, umin, umax, lbw, ubw, lbg, ubg, xmin, xmax

# ============================================================================
# SIMULACIÓN PRINCIPAL
# ============================================================================
def main():
    print("=" * 65)
    print("  MHE + NMPC — Simulación Parámetros Planta Real")
    print("  Ref: 0.10 → 0.20 → 0.15 m | T=800s | σ=1mm | rng=42")
    print("=" * 65)

    # Estados estacionarios
    print("\nCalculando estados estacionarios...")
    SS = {}
    for h2r in [0.10, 0.15, 0.20]:
        x_ss, u_ss = compute_ss(h2r)
        if x_ss is not None:
            SS[h2r] = (x_ss, u_ss)
            print(f"  h2={h2r:.2f}m → x_ss={np.round(x_ss,4)}  u_ss={np.round(u_ss,4)} ✓")
        else:
            print(f"  h2={h2r:.2f}m → SS NO encontrado ✗")

    # Modelo y solvers
    print("\nConstruyendo solvers...")
    F_rk4, nx, nu = build_casadi_model()
    (solver_mhe, N_mhe, ny, C_mat, xmin_mhe, xmax_mhe,
     lbw_mhe, ubw_mhe) = build_mhe(F_rk4, nx, nu)
    (solver_mpc, N_mpc, umin, umax,
     lbw_mpc, ubw_mpc, lbg_mpc, ubg_mpc,
     xmin_mpc, xmax_mpc) = build_nmpc(F_rk4, nx, nu)

    # Referencias
    t1 = int(Nsim*0.33); t2 = int(Nsim*0.66)
    yref_v = np.concatenate([
        0.10*np.ones(t1),
        0.20*np.ones(t2-t1),
        0.15*np.ones(Nsim-t2),
    ])
    ref_change_times_s = [t1*Ts, t2*Ts]

    # Condición inicial
    x_ss0, u_ss0 = SS[0.10]
    x0 = x_ss0.copy()

    # Historial
    T_hist = np.arange(Nsim+1)*Ts
    X_true = np.zeros((nx, Nsim+1)); X_true[:,0] = x0
    X_est  = np.zeros((nx, Nsim+1)); X_est[:,0]  = x0
    U_hist = np.zeros((nu, Nsim))
    t_mhe_ms = []; t_mpc_ms = []; t_total_ms = []

    # Buffers MHE
    Uk_buf = np.tile(u_ss0,  (N_mhe,1)).T
    Yk_buf = np.tile(C_mat@x0, (N_mhe,1)).T
    xbar   = x0.copy()
    w0_mhe = np.tile(x0, N_mhe+1)

    # Warm-start NMPC
    uprev = u_ss0.copy()
    Xw = np.tile(x0, (N_mpc+1,1)).T
    Uw = np.tile(u_ss0, (N_mpc,1)).T
    w0_mpc = np.concatenate([Xw.flatten(order='F'), Uw.flatten(order='F')])

    rng = np.random.default_rng(42)

    print(f"\nSimulando {Nsim} pasos | 0.10→0.20→0.15m | σ=1mm...")
    print(f"  {'t[s]':>6} | {'h2_real':>8} | {'h2_est':>8} | "
          f"{'ref':>6} | {'MHE':>7} | {'NMPC':>7}")
    print(f"  {'-'*55}")
    t_exp = time.time()

    for k in range(Nsim):
        t_paso = time.time()
        yr = yref_v[k]

        # Medición ruidosa
        y_noisy = C_mat @ X_true[:,k] + sigma * rng.standard_normal(ny)
        y_noisy = np.maximum(y_noisy, 0)

        # ── MHE ──────────────────────────────────────────────────────
        Uk_buf = np.roll(Uk_buf, -1, axis=1); Uk_buf[:,-1] = uprev
        Yk_buf = np.roll(Yk_buf, -1, axis=1); Yk_buf[:,-1] = y_noisy

        par_mhe = np.concatenate([xbar,
                                   Uk_buf.flatten(order='F'),
                                   Yk_buf.flatten(order='F')])
        t0 = time.time()
        try:
            sol_mhe = solver_mhe(x0=w0_mhe, lbx=lbw_mhe, ubx=ubw_mhe, p=par_mhe)
            X_opt   = np.array(sol_mhe['x']).flatten().reshape(N_mhe+1, nx).T
            x_hat   = np.clip(X_opt[:,-1], 0, H_MAX)
            xbar    = np.clip(X_opt[:,1],  0, H_MAX)
            w0_mhe  = np.concatenate([X_opt[:,1:].flatten(order='F'),
                                       X_opt[:,-1]])
        except:
            x_hat = xbar.copy()
        t_mhe_ms.append((time.time()-t0)*1000)
        X_est[:,k] = x_hat

        # ── NMPC ─────────────────────────────────────────────────────
        # Warm-start al cambio de referencia
        if k > 0 and yref_v[k] != yref_v[k-1] and yr in SS:
            x_ss_new, u_ss_new = SS[yr]
            Xw = np.tile(x_ss_new, (N_mpc+1,1)).T
            Uw = np.tile(u_ss_new, (N_mpc,1)).T
            w0_mpc = np.concatenate([Xw.flatten(order='F'), Uw.flatten(order='F')])

        par_mpc = np.concatenate([x_hat, [yr], uprev])
        t0 = time.time()
        try:
            sol_mpc = solver_mpc(x0=w0_mpc, lbx=lbw_mpc, ubx=ubw_mpc,
                                  lbg=lbg_mpc, ubg=ubg_mpc, p=par_mpc)
            w_opt   = np.array(sol_mpc['x']).flatten()
            Xtraj   = w_opt[:nx*(N_mpc+1)].reshape(N_mpc+1, nx).T
            Utraj   = w_opt[nx*(N_mpc+1):].reshape(N_mpc, nu).T
            uk      = np.clip(Utraj[:,0], umin, umax)
            Xsh = np.hstack([Xtraj[:,1:], Xtraj[:,-1:]])
            Ush = np.hstack([Utraj[:,1:], Utraj[:,-1:]])
            w0_mpc = np.concatenate([Xsh.flatten(order='F'), Ush.flatten(order='F')])
        except:
            uk = uprev.copy()
        t_mpc_ms.append((time.time()-t0)*1000)

        t_total_ms.append((time.time()-t_paso)*1000)
        U_hist[:,k] = uk; uprev = uk.copy()

        # ── Planta ───────────────────────────────────────────────────
        X_true[:,k+1] = rk4_py(X_true[:,k], uk)

        if k % 50 == 0:
            print(f"  {k*Ts:>6.0f} | {X_true[1,k+1]*100:>8.3f} | "
                  f"{x_hat[1]*100:>8.3f} | {yr*100:>6.1f} | "
                  f"{t_mhe_ms[-1]:>6.0f}ms | {t_mpc_ms[-1]:>6.0f}ms")

    X_est[:,Nsim] = X_est[:,Nsim-1]
    print(f"\n  Completado en {time.time()-t_exp:.1f}s")

    # ── Métricas ──────────────────────────────────────────────────────────
    ref_full = np.append(yref_v, yref_v[-1])
    mask_400 = T_hist > 400
    settling_mask = np.ones(len(T_hist), dtype=bool)
    for tc in ref_change_times_s:
        settling_mask &= ~((T_hist >= tc) & (T_hist < tc + SETTLING_S))
    mask_ss = mask_400 & settling_mask

    e1     = (X_true[0,:Nsim] - X_est[0,:Nsim]) * 100
    e2     = (X_true[1,:Nsim] - X_est[1,:Nsim]) * 100
    e3     = (X_true[2,:Nsim] - X_est[2,:Nsim]) * 100
    e_ctrl = (X_true[1,:] - ref_full) * 100

    rmse_e1_ss   = np.sqrt(np.mean(e1[mask_ss[:Nsim]]**2))
    rmse_e2_ss   = np.sqrt(np.mean(e2[mask_ss[:Nsim]]**2))
    rmse_e3_ss   = np.sqrt(np.mean(e3[mask_ss[:Nsim]]**2))
    rmse_ctrl_ss = np.sqrt(np.mean(e_ctrl[mask_ss]**2))
    rmse_ctrl_400 = np.sqrt(np.mean(e_ctrl[mask_400]**2))

    avg_mhe  = np.mean(t_mhe_ms);   p99_mhe  = np.percentile(t_mhe_ms, 99)
    avg_mpc  = np.mean(t_mpc_ms);   p99_mpc  = np.percentile(t_mpc_ms, 99)
    avg_tot  = np.mean(t_total_ms); p99_tot  = np.percentile(t_total_ms, 99)

    print(f"\n{'='*60}")
    print(f"  MÉTRICAS — Simulación Planta Real (RMSE_SS, excl. {SETTLING_S}s)")
    print(f"{'='*60}")
    print(f"  RMSE est.  h1 = {rmse_e1_ss:.4f} cm")
    print(f"  RMSE est.  h2 = {rmse_e2_ss:.4f} cm  (NO medido)")
    print(f"  RMSE est.  h3 = {rmse_e3_ss:.4f} cm")
    print(f"  RMSE ctrl  h2 = {rmse_ctrl_ss:.4f} cm  ← comparable con Koopman")
    print(f"  RMSE ctrl (t>400s incl.trans.) = {rmse_ctrl_400:.4f} cm")
    print(f"\n  Tiempos de cómputo:")
    print(f"  {'Componente':<18} {'Media [ms]':>12}  {'P99 [ms]':>10}")
    print(f"  {'-'*44}")
    print(f"  {'MHE':<18} {avg_mhe:>12.1f}  {p99_mhe:>10.1f}")
    print(f"  {'NMPC':<18} {avg_mpc:>12.1f}  {p99_mpc:>10.1f}")
    print(f"  {'TOTAL MHE+NMPC':<18} {avg_tot:>12.1f}  {p99_tot:>10.1f}")

    # Tabla comparativa — valores de koopman_closed_loop.ipynb
    koopman_ctrl_ss = 0.0266   # cm — de koopman_closed_loop
    koopman_est_h2  = 0.0348   # cm — de koopman_closed_loop
    koopman_t_media = 8.0      # ms — de koopman_closed_loop
    koopman_t_p99   = 14.4     # ms — de koopman_closed_loop

    print(f"\n{'='*65}")
    print(f"  TABLA COMPARATIVA — MHE+NMPC vs Koopman Closed-Loop")
    print(f"{'='*65}")
    print(f"  {'Métrica':<38} {'MHE+NMPC':>10} {'Koopman':>10}")
    print(f"  {'-'*60}")
    print(f"  {'RMSE est. h2 (SS) [cm]':<38} {rmse_e2_ss:>10.4f} {koopman_est_h2:>10.4f}")
    print(f"  {'RMSE ctrl h2 (SS) [cm]':<38} {rmse_ctrl_ss:>10.4f} {koopman_ctrl_ss:>10.4f}")
    print(f"  {'Tiempo medio/paso [ms]':<38} {avg_tot:>10.1f} {koopman_t_media:>10.1f}")
    print(f"  {'Tiempo P99/paso [ms]':<38} {p99_tot:>10.1f} {koopman_t_p99:>10.1f}")
    if koopman_t_media > 0:
        speedup = avg_tot / koopman_t_media
        delta_ctrl = (rmse_ctrl_ss - koopman_ctrl_ss) / koopman_ctrl_ss * 100
        print(f"\n  MHE+NMPC es {speedup:.1f}× más lento que Koopman | "
              f"degradación ctrl: {delta_ctrl:+.1f}%")

    # ── Gráficas ───────────────────────────────────────────────────────────
    fig, axes = plt.subplots(3, 2, figsize=(13, 11))
    fig.suptitle(
        f'MHE+NMPC — Simulación\n'
        f'RMSE_SS ctrl={rmse_ctrl_ss:.3f}cm | est h₂={rmse_e2_ss:.3f}cm | '
        f'MHE={avg_mhe:.0f}ms | NMPC={avg_mpc:.0f}ms | Total={avg_tot:.0f}ms/paso',
        fontsize=9, fontweight='bold')

    ax = axes[0, 0]
    ax.plot(T_hist, X_true[0]*100, 'b', lw=1.5, label='$h_1$ real')
    ax.plot(T_hist, X_est[0]*100,  'r--', lw=1.5, label='$h_1$ est (MHE)')
    ax.set_ylabel('$h_1$ [cm]'); ax.set_xlabel('t [s]')
    ax.set_title('Tanque 1'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[0, 1]
    ax.plot(T_hist, X_true[1]*100, 'b', lw=2,   label='$h_2$ real')
    ax.plot(T_hist, X_est[1]*100,  'r--', lw=2,  label='$h_2$ est (MHE)')
    ax.stairs(ref_full[:-1]*100, T_hist, color='k', linestyle='--', lw=1.5,
              label='Referencia')
    ax.set_ylabel('$h_2$ [cm]'); ax.set_xlabel('t [s]')
    ax.set_title('Tanque 2 — NO medido'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[1, 0]
    ax.plot(T_hist, X_true[2]*100, 'b', lw=1.5, label='$h_3$ real')
    ax.plot(T_hist, X_est[2]*100,  'r--', lw=1.5, label='$h_3$ est (MHE)')
    ax.set_ylabel('$h_3$ [cm]'); ax.set_xlabel('t [s]')
    ax.set_title('Tanque 3'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    e_ctrl_plot = (X_true[1, :] - ref_full) * 100
    rmse_ctrl_ss_plot = float(np.sqrt(np.mean(e_ctrl_plot[mask_ss]**2)))
    bias_ctrl_ss_plot = float(np.mean(e_ctrl_plot[mask_ss]))
    ax = axes[1, 1]
    ax.plot(T_hist, e_ctrl_plot, 'r', lw=1.5,
            label=f'RMSE_SS={rmse_ctrl_ss_plot:.3f}cm | Bias={bias_ctrl_ss_plot:.3f}cm')
    ax.axhline(0, color='k', ls=':')
    ax.axvline(400, color='gray', lw=0.8, ls=':', label='t=400s')
    for i, tc in enumerate(ref_change_times_s):
        ax.axvspan(tc, tc + SETTLING_S, alpha=0.12, color='orange',
                   label='transitorio excluido' if i == 0 else None)
    ax.set_ylabel('Error ctrl $h_2$ [cm]'); ax.set_xlabel('t [s]')
    ax.set_title('Error de seguimiento'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[2, 0]
    ax.stairs(U_hist[0], T_hist, color='b', lw=1.5, label='$u_1$')
    ax.stairs(U_hist[1], T_hist, color='r', lw=1.5, label='$u_3$')
    ax.set_ylim([-0.05, 1.1]); ax.set_ylabel('Control [0–1]')
    ax.set_xlabel('t [s]'); ax.set_title('Señales de control')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[2, 1]
    ax.plot(T_hist[:Nsim], t_mhe_ms,   'b', lw=0.8, alpha=0.7,
            label=f'MHE ({avg_mhe:.0f}ms)')
    ax.plot(T_hist[:Nsim], t_mpc_ms,   'r', lw=0.8, alpha=0.7,
            label=f'NMPC ({avg_mpc:.0f}ms)')
    ax.plot(T_hist[:Nsim], t_total_ms, 'k', lw=1.0, alpha=0.8,
            label=f'Total ({avg_tot:.0f}ms)')
    ax.axhline(avg_tot, color='k', lw=1.5, ls='--',
               label=f'Media={avg_tot:.0f}ms')
    ax.axhline(p99_tot, color='r', lw=1.5, ls=':',
               label=f'P99={p99_tot:.0f}ms')
    ax.set_ylabel('Tiempo [ms]'); ax.set_xlabel('Paso k')
    ax.set_title('Tiempo de cómputo por paso'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    fname = output_dir / 'mhe_nmpc_closed_loop_planta_real.png'
    plt.savefig(fname, dpi=200, bbox_inches='tight'); plt.close()
    print(f"\nFigura guardada: {fname}")

if __name__ == '__main__':
    main()

  MHE + NMPC — Simulación Parámetros Planta Real
  Ref: 0.10 → 0.20 → 0.15 m | T=800s | σ=1mm | rng=42

Calculando estados estacionarios...
  h2=0.10m → x_ss=[0.114  0.1    0.0816]  u_ss=[0.3293 0.    ] ✓
  h2=0.15m → x_ss=[0.1676 0.15   0.1267]  u_ss=[0.4039 0.    ] ✓
  h2=0.20m → x_ss=[0.2208 0.2    0.1725]  u_ss=[0.4669 0.    ] ✓

Construyendo solvers...
  Construyendo MHE...
  MHE listo (N=20).
  Construyendo NMPC...
  NMPC listo (N=20, q=500).

Simulando 400 pasos | 0.10→0.20→0.15m | σ=1mm...
    t[s] |  h2_real |   h2_est |    ref |     MHE |    NMPC
  -------------------------------------------------------
       0 |    9.999 |   10.002 |   10.0 |      4ms |     10ms
     100 |   10.006 |    9.980 |   10.0 |      4ms |      9ms
     200 |    9.971 |    9.989 |   10.0 |      4ms |      9ms
     300 |   19.408 |   19.538 |   20.0 |      6ms |      7ms
     400 |   20.002 |   19.993 |   20.0 |      4ms |      7ms
     500 |   19.992 |   19.968 |   20.0 |      4ms |      8ms
     60